<a href="https://colab.research.google.com/github/jppeirce/DSC210-Foundations-of-Data-Science/blob/main/Homework/HW05-unsupervised_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 5: Finding Groups in River Water

**DSC 210 Foundations of Data Science**

*Covers Module 9 (unsupervised learning).*

```
ASK  ->  GET  ->  EXPLORE  ->  [ MODEL ]  ->  COMMUNICATE
```

References:
- Module 9 course notebook (k-means, choosing k, hierarchical clustering, DBSCAN, PCA)
- Huber, *Hands-on Introduction to Data Science with Python* (CC BY-NC-SA 4.0): [Clustering](https://florian-huber.github.io/data_science_course/notebooks/12_clustering.html) and [Clustering beyond the basics](https://florian-huber.github.io/data_science_course/notebooks/12b_clustering_beyond_the_basics.html) (Ward linkage)
- Water quality data: summary statistics from the Upper Mississippi River Restoration Program's Long Term Resource Monitoring (LTRM) element, Pool 8, `csv_data/pool8water.csv`

*Last major revision: 2026-10-25*

## Key Concepts

- Carry out k-means by hand in two dimensions, and check it in code
- Explain why clustering needs **scaled** features, and see what goes wrong without scaling
- Choose `k`, read clusters through their centroids, and picture them with PCA
- Compare k-means with hierarchical (Ward) clustering and DBSCAN
- Match algorithms to the shapes of data they can and cannot find
- Report what a clustering does and does not show, including that every algorithm finds clusters even in pure noise

## Instructions

**Total: 50 points.** Point values are listed with each problem.

- Cells marked `# RUN-TOGETHER` are complete; run them and read the output.
- Cells marked `# FILL-IN` contain blanks written as `____`. They raise an error until every blank is replaced. Lines marked `# YOUR CODE` are yours to write.
- Run the cells **in order**. Later problems use the table `water` and the scaled array `X` from Part A.
- Every figure must include a title and labeled axes.
- Most of the credit rides on your written answers. Quote specific numbers from your output.
- Submit the completed `.ipynb`, with all cells run and all figures visible, to Canvas.

---
## Part 0. The assignment and the data (2 points)
---

**The scenario.** You are interning with river scientists who monitor Pool 8 of the Mississippi River, the stretch that runs past La Crosse. Four times a year they sample the water in four kinds of habitat. A biologist asks:

> *"If you ignored the calendar, would the water measurements alone sort these samples into seasons? And which samples don't fit any group?"*

This is a natural question for **unsupervised learning**: the seasons are labels we *have*, but we will hide them from every algorithm, cluster on the measurements alone, and only then compare, exactly as Module 9 did with the penguin species.

**About the data.** Each row of `pool8water.csv` is **one summary statistic** (the median, a percentile, or the number of samples `n`) for one combination of year, season, and habitat. We use only the **medians**.

| Column | Meaning |
| --- | --- |
| `YEAR` | year sampled (1993 to 2025) |
| `SEASON` | sampling season: `SP` spring, `SM` summer, `FL` fall, `WN` winter |
| `STRATUM` | habitat: `MC` main channel, `SC` side channel, `BWC` contiguous backwater, `IMP` impounded area, `POOL` the whole pool |
| `STAT` | which summary statistic the row holds: `median`; `p10`, `p25`, `p75`, `p90` (the 10th, 25th, 75th, and 90th percentiles); or `n`, the number of measurements the summary is based on |
| `TEMP` | water temperature (degrees C) |
| `DO` | dissolved oxygen (mg/L) |
| `COND` | specific conductance (microsiemens per cm) |
| `PH` | pH |
| `TURB` | turbidity (NTU) |
| `SS` | suspended solids (mg/L) |
| `CHLCAL` | chlorophyll *a*, a measure of algae (micrograms per L) |
| `TN`, `TP` | total nitrogen and total phosphorus (mg/L) |

In [ ]:
# RUN-TOGETHER  (setup)
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

url = 'https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/csv_data/pool8water.csv'
pool8 = pd.read_csv(url)
print('shape:', pool8.shape)
pool8.head(8)

In [ ]:
# RUN-TOGETHER  (Problem 0): what kinds of rows are there?
print(pool8['STAT'].value_counts())

# All rows for one year, season, and habitat: fall 1993, contiguous backwater
one = pool8[(pool8['YEAR'] == 1993) & (pool8['SEASON'] == 'FL') & (pool8['STRATUM'] == 'BWC')]
one[['YEAR', 'SEASON', 'STRATUM', 'STAT', 'TEMP', 'DO', 'TURB']]

**Problem 0.** (2 points)

(a) Report the shape. How many kinds of row does the `STAT` column hold, and how many rows of each kind?

(b) The six rows in the last output all describe the same water: the backwater in fall 1993. Using the `STAT` column and the data table above, explain what each of these `TEMP` values means: 15.30, 60.00, and 16.20.

(c) Why would it be a mistake to compute the average `TEMP` of the whole table?

**Your answer:**

(a)

(b)

(c)

---
## Part A. k-means, by hand and on the river (10 points)
---

### Problem A1. k-means by hand in two dimensions (3 points)

Module 9 ran k-means by hand on a line. Here are six points in a plane, with `k = 2`, and a deliberately **bad** start: both centroids in the same corner.

| point | P1 | P2 | P3 | P4 | P5 | P6 |
| --- | --- | --- | --- | --- | --- | --- |
| (x, y) | (1, 1) | (2, 1) | (1, 2) | (6, 5) | (7, 5) | (6, 6) |

Starting centroids: $c_1 = (1, 1)$ and $c_2 = (2, 1)$. Use **Euclidean** distance, $\sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2}$.

**(a) Iteration 1.** Fill in the distances (two decimals) and assignments. Then compute the new centroids, and the WCSS of this assignment measured from the new centroids.

| point | distance to $c_1$ | distance to $c_2$ | cluster |
| --- | --- | --- | --- |
| P1 (1, 1) |  |  |  |
| P2 (2, 1) |  |  |  |
| P3 (1, 2) |  |  |  |
| P4 (6, 5) |  |  |  |
| P5 (7, 5) |  |  |  |
| P6 (6, 6) |  |  |  |

New $c_1$ = ________  New $c_2$ = ________  WCSS = ________

**(b) Iteration 2.** Using the new centroids, which point changes cluster? (You only need to check the point that sits closest to the other group.) Give the centroids after this change.

**(c)** Does iteration 3 change anything? What did the bad start cost, and which `KMeans` argument protects against bad starts?

**Your answer:**

(a)

(b)

(c)

In [ ]:
# RUN-TOGETHER  (Problem A1): check your final answer
P = np.array([[1, 1], [2, 1], [1, 2], [6, 5], [7, 5], [6, 6]], dtype=float)
start = np.array([[1, 1], [2, 1]], dtype=float)

km_hand = KMeans(n_clusters=2, init=start, n_init=1).fit(P)   # init= sets the starting centroids
print('labels    :', km_hand.labels_)
print('centroids :', km_hand.cluster_centers_.round(3).tolist())
print('WCSS      :', round(km_hand.inertia_, 3))
print('iterations:', km_hand.n_iter_)

### Problem A2. Get the rows that mean something (3 points)

Keep only the **median** rows, drop the `POOL` rows (they appear to summarize the whole pool, so keeping them would count the same water twice), and drop rows with any missing value in the nine measurement columns.

In [ ]:
# FILL-IN  (Problem A2)
features = ['TEMP', 'DO', 'COND', 'PH', 'TURB', 'SS', 'CHLCAL', 'TN', 'TP']

medians = pool8[pool8['STAT'] == ____]
water = medians[medians['STRATUM'] ____ 'POOL']
water = water.dropna(subset=____).copy()

print('median rows      :', len(medians))
print('without POOL     :', len(medians[medians['STRATUM'] != 'POOL']))
print('after dropping NA:', len(water))
print(pd.crosstab(water['STRATUM'], water['SEASON']))

(a) In one sentence: what does **one row** of `water` represent?

(b) How many rows remain at each step? Use the crosstab: is the design roughly balanced across seasons and habitats?

**Your answer:**

(a)

(b)

### Problem A3. Scale first, or not? (4 points)

The biologist's first guess is that the water has two states: winter and everything else. Run k-means with `k = 2` twice, on the raw measurements and on scaled ones, and compare each clustering with the seasons the algorithm never saw.

In [ ]:
# FILL-IN  (Problem A3)
# (i) Raw, unscaled measurements
km_raw = KMeans(n_clusters=2, n_init=10, random_state=0).fit(water[features])
water['raw2'] = km_raw.labels_
print(pd.crosstab(water['raw2'], water['SEASON']))
print(water.groupby('raw2')[features].median().round(1))

In [ ]:
# FILL-IN  (Problem A3 continued)
# (ii) Scaled measurements: mean 0 and standard deviation 1 in every column
X = StandardScaler().fit_transform(water[____])

km2 = KMeans(n_clusters=____, n_init=10, random_state=0).fit(X)
water['scaled2'] = km2.labels_
# YOUR CODE: the crosstab of scaled2 against SEASON, and the median of each feature by scaled2

(a) Does the **unscaled** clustering separate winter from the other seasons? Which single feature differs most between its two clusters, and why does that feature dominate the distance? (Hint: compare the ranges of the columns.)

(b) Does the **scaled** clustering separate winter? Quote the crosstab.

(c) Using the medians, describe the winter cluster in words. Which measurements set it apart?

**Your answer:**

(a)

(b)

(c)

---
## Part B. More than two groups (10 points)
---

### Problem B1. How many clusters? (3 points)

In [ ]:
# FILL-IN  (Problem B1)
ks = range(1, 9)
wcss = [KMeans(n_clusters=k, n_init=10, random_state=0).fit(____).inertia_ for k in ks]

for k, w in zip(ks, wcss):
    print(f'k = {k}   WCSS = {w:7.1f}')

sns.lineplot(x=list(ks), y=wcss, marker='o')
plt.title('Elbow plot: Pool 8 water, nine scaled measurements')
plt.xlabel('k (number of clusters)')
plt.ylabel('WCSS (inertia)')
plt.xticks(list(ks))
plt.show()

(a) Compute the drop in WCSS from k = 1 to 2, 2 to 3, and 3 to 4. Which drop is largest?

(b) Is there one clear elbow? Compare this plot with the penguin elbow in Module 9. What does a gradual curve suggest about the structure of the data?

(c) There are four seasons. Give one reason, from what you know about rivers through the year, why four clusters need not line up with four seasons.

**Your answer:**

(a)

(b)

(c)

### Problem B2. Reading four clusters (4 points)

In [ ]:
# FILL-IN  (Problem B2)
km4 = KMeans(n_clusters=4, n_init=10, random_state=0).fit(X)
water['cluster'] = km4.labels_

print(water['cluster'].value_counts().sort_index())
print(pd.crosstab(water['cluster'], water['____']))
print(water.groupby('cluster')[features].____().round(2))

(a) Which cluster is the winter cluster? Is it essentially the same group you found in A3?

(b) For each of the other three clusters, name the season(s) it draws mostly from and the measurements that set it apart. Then give each cluster a short descriptive name. (Naming a cluster is *interpretation*: say what the numbers show, not what causes them.)

**Your answer:**

(a)

(b)

### Problem B3. Picture the clusters with PCA (3 points)

Nine dimensions cannot be drawn. Reduce the scaled data to two principal components, then draw the same points twice: colored by cluster, and colored by season.

In [ ]:
# FILL-IN  (Problem B3)
pca = PCA(n_components=2).fit(X)
pcs = pca.transform(X)
water['PC1'] = pcs[:, 0]
water['PC2'] = pcs[:, 1]
print('variance explained:', pca.explained_variance_ratio_.round(3), '| together:', round(pca.explained_variance_ratio_.sum(), 3))

sns.scatterplot(data=water, x='PC1', y='PC2', hue=____, palette='tab10', s=20)
plt.title('Pool 8 water in two principal components, colored by k-means cluster')
plt.show()

# YOUR CODE: the same scatterplot colored by SEASON, fully labeled

(a) How much of the variation do the two components keep together?

(b) Compare the two pictures. Which season stands apart in its own region, and which seasons overlap?

**Your answer:**

(a)

(b)

---
## Part C. Other ways to cluster the same water (10 points)
---

### Problem C1. Hierarchical clustering with Ward linkage (4 points)

Module 9 built a dendrogram with **single** linkage. Huber's chapter uses **Ward** linkage, which merges the two clusters whose union increases the within-cluster sum of squares the least. It favors compact clusters, much like k-means, but builds the whole tree first.

In [ ]:
# FILL-IN  (Problem C1)
Z = linkage(X, method=____)

# 488 leaves are too many to read, so show only the last 12 merges.
dendrogram(Z, truncate_mode='lastp', p=12)
plt.title('Ward dendrogram, Pool 8 water (last 12 merges)')
plt.xlabel('cluster (number of rows in parentheses)')
plt.ylabel('merge height')
plt.show()
print('heights of the last five merges:', Z[-5:, 2].round(1))

water['ward2'] = fcluster(Z, 2, criterion='maxclust')   # cut the tree into 2 clusters
water['ward4'] = fcluster(Z, ____, criterion='maxclust')
print(pd.crosstab(water['ward2'], water['SEASON']))
print(pd.crosstab(water['ward4'], water['cluster']))

(a) The last merge is much higher than the ones before it. What does that gap say, and what does the two-cluster cut contain?

(b) Compare the four-cluster Ward cut with the four k-means clusters. Which group do the two methods agree on exactly, and where do they disagree?

**Your answer:**

(a)

(b)

### Problem C2. DBSCAN as a detector of unusual samples (3 points)

In [ ]:
# FILL-IN  (Problem C2)
for eps in [1.0, 1.5, 2.0]:
    db = DBSCAN(eps=eps, min_samples=5).fit(X)
    n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
    print(f'eps={eps}  clusters={n_clusters}  noise={(db.labels_ == -1).sum()}')

db = DBSCAN(eps=____, min_samples=5).fit(X)
water['db'] = db.labels_
show = ['YEAR', 'SEASON', 'STRATUM', 'TEMP', 'TURB', 'CHLCAL', 'TP', 'DO']
print(water[water['db'] == ____][show].sort_values('CHLCAL', ascending=False).head(8))

(a) Record the number of clusters and noise points for each `eps`. Why does DBSCAN never find the four groups that k-means reported? (Think about how DBSCAN builds a cluster by chaining neighbors.)

(b) With `eps = 1.5`, look at the noise points with the most chlorophyll. What do they have in common? Why might a biologist find this list more useful than the clusters themselves?

**Your answer:**

(a)

(b)

### Problem C3. Code review: four quiet bugs (3 points)

A classmate wrote the cell below. **Every line runs, and every answer is wrong or misleading.** For each numbered line, say what in the output should make you suspicious and what the bug is, then write a corrected version.

In [ ]:
# RUN-TOGETHER  (a classmate's code: it runs, but every answer is wrong)

# 1. Two clusters for the two water states (winter vs open water)
km_bug = KMeans(n_clusters=2, n_init=10, random_state=0).fit(water[features])
print('1.', pd.crosstab(km_bug.labels_, water['SEASON']))

# 2. How often do k-means and Ward agree?
print('2. agreement:', round((water['cluster'] == water['ward4']).mean(), 3))

# 3. How many clusters did DBSCAN find with eps = 1.5?
print('3. clusters:', len(set(water['db'])))

# 4. The best number of clusters according to the elbow plot
print('4. best k:', list(ks)[np.argmin(wcss)])

**Your answer:**

| Line | What in the output is suspicious? | What is the bug? |
| --- | --- | --- |
| 1 |  |  |
| 2 |  |  |
| 3 |  |  |
| 4 |  |  |

In [ ]:
# Problem C3: your corrected lines here (label each one 1 to 4)

---
## Part D. Which algorithm for which shape? (10 points)
---

Real data rarely tells you its shape in advance. Here are four **made-up** datasets, each built to have a known set of true groups.

In [ ]:
# RUN-TOGETHER  (Part D setup): four made-up datasets with known groups
from sklearn.datasets import make_blobs, make_circles

shapes = {}
pts, truth = make_blobs(n_samples=450, centers=3, cluster_std=0.8, random_state=210)
shapes['round blobs'] = (pts, truth, 3)

pts, truth = make_circles(n_samples=400, factor=0.4, noise=0.05, random_state=210)
shapes['rings'] = (pts, truth, 2)

pts, truth = make_blobs(n_samples=450, centers=3, cluster_std=1.0, random_state=170)
shapes['stretched blobs'] = (pts @ np.array([[0.6, -0.6], [-0.4, 0.8]]), truth, 3)

pts, truth = make_blobs(n_samples=[300, 300, 300], centers=[[0, 0], [3, 0], [1.5, 6]],
                        cluster_std=[0.3, 0.3, 1.8], random_state=210)
shapes['uneven spread'] = (pts, truth, 3)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, (pts, truth, k)) in zip(axes, shapes.items()):
    sns.scatterplot(x=pts[:, 0], y=pts[:, 1], hue=truth, palette='tab10', s=10, ax=ax, legend=False)
    ax.set_title(f'{name} (true groups: {k})')
plt.show()

### Problem D1. Predict, then test (6 points)

**(a) Predict first (2 points).** Before running the next cell, fill in the table with **Y** (finds the true groups) or **N** (does not). Each algorithm is given the true number of groups where it needs one; DBSCAN uses `eps = 0.35` on scaled data.

| dataset | k-means | Ward | DBSCAN |
| --- | --- | --- | --- |
| round blobs |  |  |  |
| rings |  |  |  |
| stretched blobs |  |  |  |
| uneven spread |  |  |  |

In [ ]:
# RUN-TOGETHER  (Problem D1): every algorithm on every dataset
fig, axes = plt.subplots(4, 3, figsize=(13, 15))
for row, (name, (pts, truth, k)) in enumerate(shapes.items()):
    Xs = StandardScaler().fit_transform(pts)
    results = {
        'k-means': KMeans(n_clusters=k, n_init=10, random_state=0).fit(Xs).labels_,
        'Ward': fcluster(linkage(Xs, method='ward'), k, criterion='maxclust'),
        'DBSCAN': DBSCAN(eps=0.35, min_samples=5).fit(Xs).labels_,
    }
    for col, (method, labels) in enumerate(results.items()):
        ax = axes[row, col]
        sns.scatterplot(x=Xs[:, 0], y=Xs[:, 1], hue=labels, palette='tab10', s=8, ax=ax, legend=False)
        n_found = len(set(labels)) - (1 if -1 in set(labels) else 0)
        ax.set_title(f'{name}: {method}\n{n_found} cluster(s) found')
plt.tight_layout()
plt.show()

**(b) Score yourself (2 points).** Fill in what actually happened, and circle every cell where your prediction was wrong.

| dataset | k-means | Ward | DBSCAN |
| --- | --- | --- | --- |
| round blobs |  |  |  |
| rings |  |  |  |
| stretched blobs |  |  |  |
| uneven spread |  |  |  |

**(c) (2 points)** Is there one column with **Y** in every row? Connect your answer to the No Free Lunch theorem from Module 9.

**Your answer:**

(c)

### Problem D2. Clusters in pure noise (4 points)

Module 9 warned: *"k-means asked for three clusters will produce three clusters from pure noise."* Test it.

In [ ]:
# FILL-IN  (Problem D2)
rng = np.random.default_rng(210)
noise = rng.uniform(0, 10, size=(300, 2))      # 300 points spread evenly over a square: no groups at all

km_noise = KMeans(n_clusters=____, n_init=10, random_state=0).fit(noise)
print('cluster sizes:', np.bincount(km_noise.labels_))
sns.scatterplot(x=noise[:, 0], y=noise[:, 1], hue=km_noise.labels_, palette='tab10', s=15)
plt.title('k-means with k = 3 on structureless data')
plt.xlabel('x'); plt.ylabel('y')
plt.show()

wcss_noise = [KMeans(n_clusters=k, n_init=10, random_state=0).fit(noise).inertia_ for k in ks]
print('drops:', np.round(-np.diff(wcss_noise), 0))
# YOUR CODE: the elbow plot for wcss_noise, fully labeled

(a) How large are the three "clusters"? Would the colored scatterplot alone convince a reader that the groups are real?

(b) Does the elbow plot for pure noise have a bend? What does this tell you about using an elbow plot as **proof** that clusters exist?

(c) The Pool 8 winter cluster passed several checks that this noise would fail. Name one.

**Your answer:**

(a)

(b)

(c)

---
## Part E. Report to the biologist (8 points)
---

Write for the biologist, who knows rivers well but has never run a clustering algorithm.

**(a) (4 points) Answer the question.** In 3 to 5 sentences: do the water measurements alone sort the samples into seasons? Quote specific numbers from at least two methods.

**(b) (2 points) Method.** Which method would you show the biologist first, and why? One or two sentences.

**(c) (2 points) Two cautions.** Name two things the biologist should keep in mind before trusting these clusters (for example, about scaling, the choice of k, or what one row of the data is).

**Your answer:**

(a)

(b)

(c)